# k

In [ ]:
"""
Retail Banking: Account Portfolio & Ledger Transaction Dashboard
Single-file Python app: SQLite database + a GUI that works everywhere.

  * Local VS Code / .py file / local Jupyter -> a Tkinter desktop window opens.
  * Google Colab / any machine with no display -> the same dashboard renders
    inline with ipywidgets (no display driver needed, so nothing can crash).

Both front ends share one Database class, so the business rules are identical.
"""

import math
import sqlite3
import sys
from datetime import date

ACCOUNT_TYPES = ("Checking", "Savings")
KINDS = {"Close a Full Customer Profile": "customer", "Close a Single Account": "account"}
TAB_TITLES = ("Master Auditing Ledger", "Customers", "Accounts", "Transactions")
LEDGER_COLS = ("Account ID", "Customer Name", "Account Type", "Transaction Amount", "Transaction Type")


# ==========================================
# DATABASE + BUSINESS RULES
# ==========================================

def parse_amount(text):
    """Return a positive float rounded to cents, or raise ValueError."""
    try:
        v = float(text.strip().replace(",", ""))
    except ValueError:
        raise ValueError("Transaction Amount must be a number (e.g. 250 or 99.50).")
    if not math.isfinite(v):
        raise ValueError("Transaction Amount must be a finite number.")
    if v < 0:
        raise ValueError("Transaction Amount cannot be negative.")
    if round(v, 2) == 0:
        raise ValueError("Transaction Amount must be greater than zero.")
    return round(v, 2)


class Database:
    def __init__(self, path="bank.db"):
        self.con = sqlite3.connect(path)
        self.con.execute("PRAGMA foreign_keys = ON")  # required for ON DELETE CASCADE
        self.cur = self.con.cursor()
        self.cur.executescript(
            """
            CREATE TABLE IF NOT EXISTS customers(
                customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL UNIQUE COLLATE NOCASE,
                tax_id TEXT NOT NULL
            );
            CREATE TABLE IF NOT EXISTS accounts(
                account_id INTEGER PRIMARY KEY AUTOINCREMENT,
                customer_id INTEGER NOT NULL,
                account_type TEXT NOT NULL CHECK (account_type IN ('Checking', 'Savings')),
                open_date TEXT NOT NULL,
                FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ON DELETE CASCADE
            );
            CREATE TABLE IF NOT EXISTS transactions(
                transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
                account_id INTEGER NOT NULL,
                amount REAL NOT NULL CHECK (amount > 0),
                transaction_type TEXT NOT NULL CHECK (transaction_type IN ('Deposit', 'Withdrawal')),
                FOREIGN KEY (account_id) REFERENCES accounts(account_id) ON DELETE CASCADE
            );
            """
        )
        self.con.commit()
        if self.cur.execute("SELECT COUNT(*) FROM customers").fetchone()[0] == 0:
            self.seed()

    def seed(self):
        sample = [
            ("Asha Verma", "***-**-1001", [
                ("Checking", "2026-01-12", [("Deposit", 2500.00), ("Withdrawal", 300.00)]),
                ("Savings", "2026-02-03", [("Deposit", 10000.00)])]),
            ("Rohan Mehta", "***-**-1002", [
                ("Checking", "2026-03-18", [("Deposit", 1800.50), ("Withdrawal", 250.75)])]),
            ("Priya Nair", "***-**-1003", [
                ("Savings", "2026-04-22", [("Deposit", 5000.00), ("Withdrawal", 1200.00)])]),
        ]
        for cname, tax, accounts in sample:
            cid = self.cur.execute(
                "INSERT INTO customers (name, tax_id) VALUES (?, ?)", (cname, tax)).lastrowid
            for atype, opened, txns in accounts:
                aid = self.cur.execute(
                    "INSERT INTO accounts (customer_id, account_type, open_date) VALUES (?, ?, ?)",
                    (cid, atype, opened)).lastrowid
                self.cur.executemany(
                    "INSERT INTO transactions (account_id, amount, transaction_type) VALUES (?, ?, ?)",
                    [(aid, amt, ttype) for ttype, amt in txns])
        self.con.commit()

    # Teller insert: reuse the customer if they exist, otherwise create the profile first.
    # Each submission opens ONE new account with its opening deposit (all or nothing).
    def open_account(self, name, account_type, amount_text):
        name, account_type, amount_text = name.strip(), account_type.strip().title(), amount_text.strip()
        if not (name and account_type and amount_text):
            raise ValueError("Please fill in all the details.")
        if account_type not in ACCOUNT_TYPES:
            raise ValueError("Account Type must be Checking or Savings.")
        amount = parse_amount(amount_text)
        try:
            row = self.cur.execute(
                "SELECT customer_id, name FROM customers WHERE name=?", (name,)).fetchone()
            if row:
                cid, name, is_new = row[0], row[1], False
            else:
                cid = self.cur.execute(
                    "INSERT INTO customers (name, tax_id) VALUES (?, ?)",
                    (name, "NOT PROVIDED")).lastrowid
                is_new = True
            aid = self.cur.execute(
                "INSERT INTO accounts (customer_id, account_type, open_date) VALUES (?, ?, ?)",
                (cid, account_type, date.today().isoformat())).lastrowid
            self.cur.execute(
                "INSERT INTO transactions (account_id, amount, transaction_type) VALUES (?, ?, 'Deposit')",
                (aid, amount))
            self.con.commit()
        except sqlite3.Error:
            self.con.rollback()
            raise
        who = f"new customer {name}" if is_new else f"existing customer {name}"
        return f"Opened {account_type} account #{aid} for {who} with an opening deposit of {amount:,.2f}."

    # Teller delete: children disappear through ON DELETE CASCADE.
    def close(self, kind, item_id):
        if kind == "customer":
            row = self.cur.execute(
                "SELECT name FROM customers WHERE customer_id=?", (item_id,)).fetchone()
            if not row:
                raise ValueError("That customer no longer exists.")
            n_acc, n_txn = self.cur.execute(
                "SELECT COUNT(DISTINCT a.account_id), COUNT(t.transaction_id) FROM accounts a "
                "LEFT JOIN transactions t ON t.account_id = a.account_id "
                "WHERE a.customer_id=?", (item_id,)).fetchone()
            self.cur.execute("DELETE FROM customers WHERE customer_id=?", (item_id,))
            msg = (f"Closed customer {row[0]}: {n_acc} account(s) and "
                   f"{n_txn} transaction(s) were removed too.")
        else:
            row = self.cur.execute(
                "SELECT COUNT(*) FROM accounts WHERE account_id=?", (item_id,)).fetchone()
            if not row[0]:
                raise ValueError("That account no longer exists.")
            n_txn = self.cur.execute(
                "SELECT COUNT(*) FROM transactions WHERE account_id=?", (item_id,)).fetchone()[0]
            self.cur.execute("DELETE FROM accounts WHERE account_id=?", (item_id,))
            msg = f"Closed account #{item_id}: {n_txn} transaction(s) were removed too."
        self.con.commit()
        return msg

    # (id, label) pairs for the dynamic dropdown
    def choices(self, kind):
        if kind == "customer":
            sql = ("SELECT c.customer_id, c.customer_id || ' - ' || c.name || ' (' || "
                   "COUNT(a.account_id) || ' account(s))' FROM customers c "
                   "LEFT JOIN accounts a ON a.customer_id = c.customer_id "
                   "GROUP BY c.customer_id ORDER BY c.customer_id")
        else:
            sql = ("SELECT a.account_id, 'Account ' || a.account_id || ' - ' || c.name || ' / ' || "
                   "a.account_type || ' (opened ' || a.open_date || ')' FROM accounts a "
                   "JOIN customers c ON c.customer_id = a.customer_id ORDER BY a.account_id")
        return self.cur.execute(sql).fetchall()

    # The four live views: [(columns, rows), ...] in TAB_TITLES order
    def views(self):
        ledger = [(aid, cname, atype, "" if amt is None else f"{amt:,.2f}", ttype or "")
                  for aid, cname, atype, amt, ttype in self.cur.execute(
                      "SELECT a.account_id, c.name, a.account_type, t.amount, t.transaction_type "
                      "FROM accounts a JOIN customers c ON c.customer_id = a.customer_id "
                      "LEFT JOIN transactions t ON t.account_id = a.account_id "
                      "ORDER BY a.account_id, t.transaction_id").fetchall()]
        out = [(LEDGER_COLS, ledger)]
        for table in ("customers", "accounts", "transactions"):  # fixed names, never user input
            cur = self.cur.execute(f"SELECT * FROM {table}")
            out.append(([d[0] for d in cur.description], cur.fetchall()))
        return out


# ==========================================
# FRONT END 1: Tkinter desktop window
# ==========================================

GREEN, LIGHT = "#065535", "#dcf5d5"


def run_tk(bank):
    """Open the Tkinter window. Returns False if no display is available."""
    try:
        import tkinter as tk
        from tkinter import messagebox, ttk
        root = tk.Tk()
    except Exception:  # ImportError, or TclError when there is no display
        return False

    root.title("CREDIT UNION - ACCOUNT PORTFOLIO & LEDGER DASHBOARD")
    root.geometry("1100x760+40+40")
    root.config(bg=GREEN)

    name, acct_type, amount = tk.StringVar(), tk.StringVar(), tk.StringVar()
    kind = tk.StringVar(value=next(iter(KINDS)))
    choice = tk.StringVar()
    choice_ids = {}  # dropdown label -> database id

    tk.Label(root, text="ACCOUNT PORTFOLIO & LEDGER DASHBOARD",
             font=("Calibri", 18, "bold"), bg=GREEN, fg="white").pack(side="top", pady=8)

    # ---------------- Teller Insert Panel ----------------
    ins = tk.LabelFrame(root, text=" Teller Insert Panel ", font=("Calibri", 14, "bold"),
                        bg=LIGHT, fg=GREEN, padx=10, pady=6)
    ins.pack(side="top", fill="x", padx=10, pady=4)
    tk.Label(ins, text="Customer Name", font=("Calibri", 14), bg=LIGHT).grid(row=0, column=0, padx=8, pady=5, sticky="w")
    tk.Entry(ins, textvariable=name, font=("Calibri", 14), width=30).grid(row=0, column=1, padx=8, pady=5, sticky="w")
    tk.Label(ins, text="Account Type", font=("Calibri", 14), bg=LIGHT).grid(row=0, column=2, padx=8, pady=5, sticky="w")
    ttk.Combobox(ins, textvariable=acct_type, font=("Calibri", 14), width=18, state="readonly",
                 values=ACCOUNT_TYPES).grid(row=0, column=3, padx=8, pady=5, sticky="w")
    tk.Label(ins, text="Transaction Amount", font=("Calibri", 14), bg=LIGHT).grid(row=1, column=0, padx=8, pady=5, sticky="w")
    tk.Entry(ins, textvariable=amount, font=("Calibri", 14), width=30).grid(row=1, column=1, padx=8, pady=5, sticky="w")

    # ---------------- Teller Delete Panel ----------------
    dele = tk.LabelFrame(root, text=" Teller Delete Panel ", font=("Calibri", 14, "bold"),
                         bg=LIGHT, fg=GREEN, padx=10, pady=6)
    dele.pack(side="top", fill="x", padx=10, pady=4)
    tk.Label(dele, text="Classification", font=("Calibri", 14), bg=LIGHT).grid(row=0, column=0, padx=8, pady=5, sticky="w")
    combo_kind = ttk.Combobox(dele, textvariable=kind, font=("Calibri", 14), width=32,
                              state="readonly", values=tuple(KINDS))
    combo_kind.grid(row=0, column=1, padx=8, pady=5, sticky="w")
    tk.Label(dele, text="Target", font=("Calibri", 14), bg=LIGHT).grid(row=1, column=0, padx=8, pady=5, sticky="w")
    combo_choice = ttk.Combobox(dele, textvariable=choice, font=("Calibri", 14), width=70, state="readonly")
    combo_choice.grid(row=1, column=1, columnspan=3, padx=8, pady=5, sticky="w")

    # ---------------- Tabbed Display ----------------
    style = ttk.Style()
    style.configure("mystyle.Treeview", font=("Calibri", 12), rowheight=26)
    style.configure("mystyle.Treeview.Heading", font=("Calibri", 13, "bold"))
    notebook = ttk.Notebook(root)
    trees = []
    for title in TAB_TITLES:
        frame = tk.Frame(notebook, bg="#ecf0f1")
        notebook.add(frame, text=f"  {title}  ")
        tree = ttk.Treeview(frame, show="headings", style="mystyle.Treeview", selectmode="browse")
        vsb = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
        tree.configure(yscrollcommand=vsb.set)
        tree.pack(side="left", fill="both", expand=True)
        vsb.pack(side="right", fill="y")
        trees.append(tree)

    # ---------------- Event Functions ----------------
    def fill_tree(tree, cols, rows):
        tree.delete(*tree.get_children())
        tree["columns"] = tuple(cols)
        for col in cols:
            tree.heading(col, text=col)
            tree.column(col, width=160, anchor="w")
        for r in rows:
            tree.insert("", "end", values=["" if v is None else v for v in r])

    def refresh_choices(event=None):
        """Rebuild the dropdown from the database so it never points at a deleted ID."""
        pairs = bank.choices(KINDS[kind.get()])
        choice_ids.clear()
        choice_ids.update({label: i for i, label in pairs})
        labels = [label for _, label in pairs]
        combo_choice["values"] = labels
        choice.set(labels[0] if labels else "")

    def refresh_all():
        for tree, (cols, rows) in zip(trees, bank.views()):
            fill_tree(tree, cols, rows)
        refresh_choices()

    def clear_form():
        name.set("")
        acct_type.set("")
        amount.set("")

    def add_record():
        try:
            msg = bank.open_account(name.get(), acct_type.get(), amount.get())
        except ValueError as err:
            messagebox.showerror("Error in Input", str(err))
            return
        except sqlite3.Error as err:
            messagebox.showerror("Database Error", str(err))
            return
        messagebox.showinfo("Success", msg)
        clear_form()
        refresh_all()

    def delete_selected():
        label = choice.get()
        if label not in choice_ids:
            messagebox.showwarning("Selection Required", "There is nothing selected to close.")
            return
        warn = ("ALL of this customer's accounts and ledger transactions will also be deleted."
                if KINDS[kind.get()] == "customer"
                else "ALL of this account's ledger transactions will also be deleted.")
        if not messagebox.askyesno("Confirm", f"{kind.get()}:\n{label}\n\n{warn}"):
            return
        try:
            msg = bank.close(KINDS[kind.get()], choice_ids[label])
        except ValueError as err:
            messagebox.showerror("Error", str(err))
            refresh_all()
            return
        except sqlite3.Error as err:
            messagebox.showerror("Database Error", str(err))
            return
        messagebox.showinfo("Success", msg)
        clear_form()
        choice.set("")
        refresh_all()

    def make_button(parent, text, command, **grid):
        tk.Button(parent, text=text, command=command, width=24, font=("Calibri", 13, "bold"),
                  fg="#ecf0f1", bg=GREEN, bd=0, cursor="hand2").grid(**grid)

    make_button(ins, "Open Account & Deposit", add_record, row=2, column=0, padx=8, pady=6, sticky="w")
    make_button(ins, "Clear", clear_form, row=2, column=1, padx=8, pady=6, sticky="w")
    make_button(dele, "Execute Delete", delete_selected, row=2, column=1, padx=8, pady=6, sticky="w")
    notebook.pack(side="top", fill="both", expand=True, padx=10, pady=(4, 10))

    combo_kind.bind("<<ComboboxSelected>>", refresh_choices)
    root.protocol("WM_DELETE_WINDOW", root.destroy)
    refresh_all()
    root.lift()
    root.attributes("-topmost", True)  # bring the window forward when launched from Jupyter
    root.after(400, lambda: root.attributes("-topmost", False))
    root.mainloop()
    return True


# ==========================================
# FRONT END 2: inline ipywidgets (Colab / no display)
# ==========================================

def run_widgets(bank):
    try:
        import ipywidgets as w
        from IPython.display import display
    except ImportError:
        print("No desktop display and no notebook widgets found.")
        print("Run:  pip install ipywidgets   (then restart the kernel and run again)")
        return False
    from html import escape

    LBL = {"description_width": "140px"}
    WIDE = w.Layout(width="520px")
    name_in = w.Text(description="Customer Name", placeholder="e.g. Asha Verma", style=LBL, layout=WIDE)
    type_in = w.Dropdown(description="Account Type", style=LBL, layout=WIDE,
                         options=[("Select...", "")] + [(t, t) for t in ACCOUNT_TYPES], value="")
    amount_in = w.Text(description="Transaction Amount", placeholder="e.g. 250.00", style=LBL, layout=WIDE)
    add_btn = w.Button(description="Open Account & Deposit", button_style="success", icon="plus")

    kind_in = w.Dropdown(description="Classification", options=list(KINDS), style=LBL, layout=WIDE)
    choice_in = w.Dropdown(description="Target", options=[], style=LBL, layout=w.Layout(width="760px"))
    del_btn = w.Button(description="Execute Delete", button_style="danger", icon="trash")
    status = w.HTML(value="")

    views = [w.HTML() for _ in TAB_TITLES]
    tabs = w.Tab(children=views)
    if hasattr(tabs, "set_title"):
        for i, t in enumerate(TAB_TITLES):
            tabs.set_title(i, t)
    else:
        tabs.titles = tuple(TAB_TITLES)

    def render(cols, rows):
        cell = "border:1px solid rgba(128,128,128,.4);padding:4px 10px;text-align:left;"
        head = "".join(f"<th style='{cell}position:sticky;top:0;background:rgba(128,128,128,.25)'>"
                       f"{escape(str(c))}</th>" for c in cols)
        body = "".join("<tr>" + "".join(f"<td style='{cell}'>{'' if v is None else escape(str(v))}</td>"
                                        for v in r) + "</tr>" for r in rows)
        if not rows:
            body = f"<tr><td colspan='{len(cols)}' style='{cell}'><i>No rows</i></td></tr>"
        return (f"<div style='max-height:340px;overflow:auto'><table style='border-collapse:collapse;"
                f"font-size:13px'><tr>{head}</tr>{body}</table></div>")

    def refresh_choices():
        pairs = bank.choices(KINDS[kind_in.value])
        choice_in.options = [(label, i) for i, label in pairs]
        choice_in.value = pairs[0][0] if pairs else None

    def refresh_all():
        for view, (cols, rows) in zip(views, bank.views()):
            view.value = render(cols, rows)
        refresh_choices()

    def show(msg, ok=True):
        status.value = f"<b style='color:{'#2e7d32' if ok else '#c62828'}'>{escape(msg)}</b>"

    def on_add(_):
        try:
            msg = bank.open_account(name_in.value, type_in.value, amount_in.value)
        except ValueError as err:
            show(str(err), False)
            return
        except sqlite3.Error as err:
            show(f"Database error: {err}", False)
            return
        name_in.value, type_in.value, amount_in.value = "", "", ""
        refresh_all()
        show(msg)

    def on_delete(_):
        if choice_in.value is None:
            show("There is nothing selected to close.", False)
            return
        try:
            msg = bank.close(KINDS[kind_in.value], choice_in.value)
        except ValueError as err:
            refresh_all()
            show(str(err), False)
            return
        except sqlite3.Error as err:
            show(f"Database error: {err}", False)
            return
        name_in.value, type_in.value, amount_in.value = "", "", ""
        refresh_all()
        show(msg)

    add_btn.on_click(on_add)
    del_btn.on_click(on_delete)
    kind_in.observe(lambda change: refresh_choices(), names="value")

    refresh_all()
    display(w.VBox([
        w.HTML(value="<h3>Teller Insert Panel</h3>"), name_in, type_in, amount_in, add_btn,
        w.HTML(value="<h3>Teller Delete Panel</h3>"), kind_in, choice_in, del_btn,
        status, w.HTML(value="<h3>Live Data</h3>"), tabs]))
    return True


# ==========================================
# LAUNCHER
# ==========================================

def launch():
    bank = Database("bank.db")
    if "google.colab" not in sys.modules and run_tk(bank):
        bank.con.close()  # window was closed
        return
    run_widgets(bank)


if __name__ == "__main__":
    launch()